# Confirmatory & Causal Experiments (v2) on Prefix Sensitivity in a VLM

## 0. Context & Pre-registered Hypotheses
- **Mô hình:** `LLaVA-1.5-7B` (`llava-hf/llava-1.5-7b-hf`) trên MS-COCO val2014.
- **Thí nghiệm E1:** Can thiệp tiền tố teacher-forced phân tầng theo khoảng cách $k \in \{1, 2, 3\}$ và $k \ge 4$ trên $N$ ảnh mới (loại trừ pilot).
- **Thí nghiệm E2:** Kiểm tra sinh tự do (Free-generation check) xem đối tượng có thực sự bị đổi không.
- **Thí nghiệm E3:** Can thiệp nhân quả làm mờ ảnh (Blend với ảnh đen) + Kiểm soát nhiệt độ Entropy.
- **Tiêu chuẩn khoa học:** Cluster bootstrap 95% CI (2.000 lượt resamples), không dùng p-values, xuất bảng phán quyết `verdicts.md`.

> **Lưu ý:** Bạn có thể tự do tùy chỉnh số lượng ảnh `N_IMAGES` (ví dụ: 10, 30, 50, hoặc 150 ảnh) và giai đoạn chạy `STAGE`.

### Step 0: Clone hoặc Cập nhật Codebase từ GitHub

In [ ]:
import os
if not os.path.exists("run_all.py"):
    if not os.path.exists("/kaggle/working/Experiment_CTI"):
        !git clone https://github.com/ntmy12/Experiment_CTI.git /kaggle/working/Experiment_CTI
    %cd /kaggle/working/Experiment_CTI
else:
    !git pull origin main
print("Working directory:", os.getcwd())
!ls -la run_all.py config_v2.py

### Step 1: Cài đặt Thư viện Phụ thuộc (Dependencies)

In [ ]:
!pip install -q transformers accelerate pycocotools statsmodels scikit-learn spacy
!python -m spacy download en_core_web_sm -q

### Step 2: Kiểm tra GPU & Phần cứng (2 x Tesla T4)

In [ ]:
import torch, os
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB)")

### Step 3: Cấu hình và Chạy Pipeline Thí nghiệm v2
- `N_IMAGES`: Số lượng ảnh bạn muốn chạy (mặc định 150, bạn có thể chỉnh thành 10 để test nhanh hoặc 30 cho Milestone M2).
- `STAGE`: Giai đoạn muốn chạy (`all` = chạy trọn gói E1, E2, E3, Phân tích, Biểu đồ và Báo cáo; hoặc chọn `m0`, `m1`, `m2`, `e1`, `e2`, `e3`, `analysis`, `viz`, `report`).

In [ ]:
# === TÙY CHỈNH THÔNG SỐ TẠI ĐÂY ===
N_IMAGES = 150   # Thay đổi số lượng ảnh theo ý muốn (ví dụ: 10, 30, 50, 150)
STAGE = "all"    # Các lựa chọn: "all", "m0", "m1", "m2", "e1", "e2", "e3", "analysis", "viz", "report"

import os
os.environ["PYTHONUNBUFFERED"] = "1"

# Chạy với thanh tiến trình in-place cập nhật trực tiếp tại chỗ
!python -u run_all.py --stage {STAGE} --n_images {N_IMAGES}

### Step 4: Kiểm tra Cây Thư mục Kết quả & Dung lượng Artifacts

In [ ]:
import os
results_dir = "/kaggle/working/results_v2" if os.path.exists("/kaggle/working/results_v2") else "./results_v2"
if os.path.exists(results_dir):
    for root, dirs, files in os.walk(results_dir):
        level = root.replace(results_dir, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            fpath = os.path.join(root, f)
            size_kb = os.path.getsize(fpath) / 1024.0
            print(f"{subindent}{f} ({size_kb:.1f} KB)")
else:
    print(f"Thư mục {results_dir} chưa được tạo.")

### Step 5: Đọc Bảng Phán Quyết Giả Thuyết (Verdicts Table)

In [ ]:
verdicts_path = "/kaggle/working/results_v2/verdicts.md" if os.path.exists("/kaggle/working/results_v2/verdicts.md") else "./results_v2/verdicts.md"
if os.path.exists(verdicts_path):
    with open(verdicts_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print(f"Chưa có file {verdicts_path}.")

### Step 6: Xem Báo Cáo HTML Độc Lập Trực Tiếp Trong Notebook

In [ ]:
from IPython.display import HTML, display
import os

report_path = "/kaggle/working/results_v2/report.html" if os.path.exists("/kaggle/working/results_v2/report.html") else "./results_v2/report.html"
if os.path.exists(report_path):
    with open(report_path, "r", encoding="utf-8") as f:
        html_content = f.read()
    display(HTML(html_content))
else:
    print(f"Report not found at {report_path}")

### Step 7: Tải Xuống Toàn Bộ Kết Quả (File Link & Zip Archive)

In [ ]:
import shutil, os
from IPython.display import FileLink

src = "/kaggle/working/results_v2" if os.path.exists("/kaggle/working/results_v2") else "./results_v2"
dst = "/kaggle/working/v2_results" if os.path.exists("/kaggle") else "./v2_results"

if os.path.exists(src):
    shutil.make_archive(dst, 'zip', src)
    print(f"Đã tạo file zip kết quả: {dst}.zip ({os.path.getsize(dst + '.zip') / 1024 / 1024:.2f} MB)")
    display(FileLink(f"{dst}.zip"))
else:
    print(f"Chưa có thư mục {src} để nén.")